In [14]:
# Install PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")

import torch.nn as nn

import tiktoken

PyTorch version: 2.9.1


In [15]:
# Set model configuration
GPT_CONFIG_124M = {
  "vocab_size": 50257,    # Vocabulary size
  #"context_length": 1024, # Context length
  "context_length": 256, # Context length
  "emb_dim": 768,         # Embedding dimension
  "n_heads": 12,          # Number of attention heads
  "n_layers": 12,         # Number of layers
  "drop_rate": 0.1,       # Dropout rate
  "qkv_bias": False       # Query-Key-Value bias
}

In [16]:
# Previously defined
from production import GPTModel, generate_text_simple

In [17]:
# Create GPT model
model = GPTModel(GPT_CONFIG_124M)

# Set evaluation mode to disable dropout during inference.
# The opposite of eval() is train(). Append ";" to suppress model dump.
model.eval();

In [18]:
# Convert text to token IDs
def text_to_token_ids(text, tokenizer):
  encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
  encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
  return encoded_tensor

# Convert token IDs to text
def token_ids_to_text(token_ids, tokenizer):
  flat = token_ids.squeeze(0) # remove batch dimension
  return tokenizer.decode(flat.tolist())

# Test sample
torch.manual_seed(123)
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

# Use the GPT model to create logit vectors from the tokens
token_ids = generate_text_simple(
  model=model,
  idx=text_to_token_ids(start_context, tokenizer),
  max_new_tokens=10,
  context_size=GPT_CONFIG_124M["context_length"]
)

# Convert token IDs to text
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


## Measure Quality

Implement cross-entropy loss functions to minimize test generation error.

In [19]:
# Test input and expected output
inputs = torch.tensor([[16833, 3626, 6100],  # ["every effort moves",
                       [40,    1107, 588]])  # "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]])  # " really like chocolate"]

In [20]:
# Test use GPT model to generate logits
with torch.no_grad():
  logits = model(inputs)

# Get probability tensors for each logit
probas = torch.softmax(logits, dim=-1)
print(probas.shape)

# Get max probability tensors
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

torch.Size([2, 3, 50257])
Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [21]:
# Show max probablity words of tken IDs.
# The output reveals that the untrained model generates unexpected values.
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


In [22]:
# Logits are flattened over the batch dimension
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat)
print("Flattened targets:", targets_flat)

# Cross-entropy loss is the same as negative average log probability
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(f"Cross-entropy loss: {loss}")

# Perplexity is a measure of uncertainty in prediction
perplexity = torch.exp(loss)
print(f"Perplexity: {perplexity}")

Flattened logits: tensor([[ 0.1113, -0.1057, -0.3666,  ...,  0.2843, -0.8824,  0.1074],
        [-0.6109, -0.5167, -0.7613,  ...,  0.5450, -1.0319, -0.2175],
        [ 0.5707, -0.6459, -0.0701,  ...,  0.7419, -0.1806, -0.2217],
        [-0.2968,  0.1949, -0.1649,  ..., -0.4867,  0.7218, -0.1714],
        [-0.8375,  0.0612, -0.4641,  ...,  0.2327, -0.3889, -0.0770],
        [ 0.5614,  0.6919,  0.8915,  ..., -0.9472,  1.2411, -0.2056]])
Flattened targets: tensor([ 3626,  6100,   345,  1107,   588, 11311])
Cross-entropy loss: 10.793964385986328
Perplexity: 48725.8203125


In [23]:
# Read the training text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Total number of characters: {len(raw_text)}")

total_tokens = len(tokenizer.encode(raw_text))
print(f"Total number of tokens: {total_tokens}")

# Divide training and test (validation)
train_ratio = 0.90
split_idx = int(train_ratio * len(raw_text))
train_data = raw_text[:split_idx]
test_data = raw_text[split_idx:]
print(f"Split idx: {split_idx}")

Total number of characters: 20479
Total number of tokens: 5145
Split idx: 18431


In [ ]:
from production import create_dataloader_v1

torch.manual_seed(123)

# Load training
train_loader = create_dataloader_v1(
  train_data,
  batch_size=2,
  max_length=GPT_CONFIG_124M["context_length"],
  stride=GPT_CONFIG_124M["context_length"],
  drop_last=True,
  shuffle=True,
  num_workers=0
)

# Load test
test_loader = create_dataloader_v1(
  test_data,
  batch_size=2,
  max_length=GPT_CONFIG_124M["context_length"],
  stride=GPT_CONFIG_124M["context_length"],
  drop_last=False,
  shuffle=False,
  num_workers=0
)

In [26]:
# Compute cross-entropy loss
def calc_loss_batch(input_batch, target_batch, model, device):
  input_batch, target_batch = input_batch.to(device), target_batch.to(device)
  logits = model(input_batch)
  loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
  return loss

# Compute loss
def calc_loss_loader(data_loader, model, device, num_batches=None):
  total_loss = 0.
  if len(data_loader) == 0:
    return float("nan")
  elif num_batches is None:
    num_batches = len(data_loader)
  else:
    # Reduce the number of batches to match the total number of batches in the data loader
    # if num_batches exceeds the number of batches in the data loader
    num_batches = min(num_batches, len(data_loader))
  for i, (input_batch, target_batch) in enumerate(data_loader):
    if i < num_batches:
      loss = calc_loss_batch(input_batch, target_batch, model, device)
      total_loss += loss.item()
    else:
      break
  return total_loss / num_batches

In [29]:
# Check for GPU-capable
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  # Use PyTorch 2.9 or newer for stable mps results
  major, minor = map(int, torch.__version__.split(".")[:2])
  if (major, minor) >= (2, 9):
    device = torch.device("mps")
  else:
    device = torch.device("cpu")
else:
  device = torch.device("cpu")
print(f"Using {device} device")

# Assign model to GPU device
model.to(device)

# Disable gradient tracking for efficiency because we are not training, yet
with torch.no_grad():
  train_loss = calc_loss_loader(train_loader, model, device)
  val_loss = calc_loss_loader(test_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Using cuda device
Training loss: 10.987583372328016
Validation loss: 10.98110580444336


## Model Training

Train the GPT model.

In [ ]:
#
